# Define Ingestion Configuration


In [0]:
INGESTION_CONFIG = [
    {
        "source": "olist_customers_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_customers_dataset.csv",
        "table": "olist_customers_table"
    },
    {
        "source": "olist_geolocation_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_geolocation_dataset.csv",
        "table": "olist_geolocation_table"
    },
    {
        "source": "olist_order_items_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_order_items_dataset.csv",
        "table": "olist_order_items_table"
    },
    {
        "source": "olist_order_payments_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_order_payments_dataset.csv",
        "table": "olist_order_payments_table"
    },
    {
        "source": "olist_order_reviews_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_order_reviews_dataset.csv",
        "table": "olist_order_reviews_table"
    },
    {
        "source": "olist_orders_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_orders_dataset.csv",
        "table": "olist_orders_table"
    },
    {
        "source": "product_category_name_translation.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/product_category_name_translation.csv",
        "table": "product_category_name_translation_table"
    },
    {
        "source": "olist_products_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_products_dataset.csv",
        "table": "olist_products_table"
    },
    {
        "source": "olist_sellers_dataset.csv",
        "path": "/Volumes/e-commerce-lake-house-project/bronze/source-data/olist_sellers_dataset.csv",
        "table": "olist_sellers_table"
    }
]

# Ingest Files into Bronze Tables

In [0]:
from pyspark.sql.functions import current_timestamp, lit

results = []

for item in INGESTION_CONFIG:
    source = item["source"]
    path   = item["path"]
    table  = item["table"]
    target = f"`e-commerce-lake-house-project`.bronze.{table}"

    print(f"Ingesting {source}...")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .option("encoding", "UTF-8")
              .csv(path))

        # Add audit columns
        df_bronze = (df
            .withColumn("_ingested_at",  current_timestamp())
            .withColumn("_source_file",  lit(source))
            .withColumn("_batch_id",     lit("batch_001")))

        row_count = df_bronze.count()

        (df_bronze.write
            .mode("overwrite")
            .format("delta")
            .option("overwriteSchema", "true")
            .saveAsTable(target))

        results.append({"table": table, "rows": row_count, "status": "OK"})
        print(f"  Done — {row_count:,} rows written to {target}\n")

    except Exception as e:
        results.append({"table": table, "rows": 0, "status": f"FAILED: {e}"})
        print(f"  FAILED: {e}\n")

# Summary report
print("=" * 55)
print(f"{'Table':<45} {'Rows':>8}  Status")
print("-" * 55)
for r in results:
    print(f"{r['table']:<45} {r['rows']:>8,}  {r['status']}")